# 08b: Debug Notebook - 1D Array Operations (L2)

## Purpose
Validate DSL 1D array operations at Level 2 (RVec operations).

## Phase 13.6.G Debug Notebook

This notebook tests:
- 1D slicing (`track_pt[:3]`, `track_pt[-2:]`, `track_pt[::2]`)
- 1D reductions (`Sum`, `Mean`, `Max`, `Min`)
- 1D element access (`track_pt[0]`, `track_pt[-1]`)
- 1D transformations (`track_pt / 1000`, `sqrt(track_pt)`)

## Setup

In [ ]:
# Path setup - ensure RDataFrameDSL is importable
import sys
import os

# Add parent directory to path if running from examples/
notebook_dir = os.path.dirname(os.path.abspath('.'))
if 'RDataFrameDSL' not in sys.modules:
    for path in ['.', '..', notebook_dir]:
        if os.path.exists(os.path.join(path, 'RDataFrameDSL')):
            sys.path.insert(0, os.path.abspath(path))
            break

In [ ]:
import ROOT
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging
import time

from RDataFrameDSL import DSLCompiler
from RDataFrameDSL.verbosity import VERBOSE_DEFAULT, VERBOSE_FULL

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("RDataFrameDSL")

# Configure plot size
plt.rcParams['figure.figsize'] = (6, 4)

print("Setup complete")

## Generate Test Data

In [ ]:
from tests.generators.toy_nd import generate_nd_2d_root

filename = generate_nd_2d_root(size='S', seed=42)
rdf = ROOT.RDataFrame("Events", filename)

print("Available columns:", list(rdf.GetColumnNames()))
print(f"Number of events: {rdf.Count().GetValue()}")

# Schema for 1D operations
schema = {
    'event_id': 'long',
    'n_tracks': 'int',
    'event_weight': 'double',
    'track_pt': 'RVec<double>',
    'track_eta': 'RVec<double>',
}

## Section 1: 1D Slicing

### Description
Test slicing operations on RVec columns.

### What we're testing
- Basic slicing: `track_pt[:3]`, `track_pt[1:-1]`
- Negative indices: `track_pt[-2:]`
- Strided slicing: `track_pt[::2]`, `track_pt[::-1]`
- Edge cases: `track_pt[5:5]` (empty), `track_pt[:100]` (beyond end)

### If it fails
Check `_visit_Subscript()` in `ir_builder.py` for slice handling.

In [ ]:
dsl = DSLCompiler(schema)

# Basic slicing (supported patterns)
dsl.define("first_3", "track_pt[:3]")
dsl.define("last_2", "track_pt[-2:]")
dsl.define("middle", "track_pt[1:-1]")  # Phase 13.6.G: mixed negative indices

# Strided slicing
dsl.define("every_other", "track_pt[::2]")
dsl.define("reversed", "track_pt[::-1]")

print(dsl.describe_structure())

In [ ]:
# Get raw data for validation
raw = rdf.Range(5).AsNumpy(['track_pt'])

# Execute DSL
applied = dsl.apply(rdf)
result = applied.Range(5).AsNumpy(['track_pt', 'first_3', 'last_2', 'middle', 'every_other', 'reversed'])

# Validate each event
for i in range(5):
    pt = np.array(raw['track_pt'][i])  # Convert RVec to numpy
    
    # first_3
    expected = pt[:3]
    actual = np.array(result['first_3'][i])
    assert np.allclose(expected, actual), f"Event {i}: first_3 mismatch"
    
    # last_2
    expected = pt[-2:] if len(pt) >= 2 else pt
    actual = np.array(result['last_2'][i])
    assert np.allclose(expected, actual), f"Event {i}: last_2 mismatch"
    
    # middle [1:-1] - Phase 13.6.G fix
    expected = pt[1:-1] if len(pt) >= 2 else np.array([])
    actual = np.array(result['middle'][i])
    assert np.allclose(expected, actual), f"Event {i}: middle mismatch"

print("✓ 1D slicing: All validations passed")

## Section 2: 1D Reductions

### Description
Test reduction operations that collapse RVec to scalar.

### What we're testing
- `Sum(track_pt)` → total pT per event
- `Mean(track_pt)` → average pT per event
- `Max(track_pt)` → maximum pT per event
- `Min(track_pt)` → minimum pT per event

### If it fails
Check reduction function mapping in `backend_cpp.py`.

In [ ]:
dsl2 = DSLCompiler(schema)

dsl2.define("sum_pt", "Sum(track_pt)")
dsl2.define("mean_pt", "Mean(track_pt)")
dsl2.define("max_pt", "Max(track_pt)")
dsl2.define("min_pt", "Min(track_pt)")

df = dsl2.to_pandas(rdf, ['sum_pt', 'mean_pt', 'max_pt', 'min_pt'])

# Validate against raw data
raw_all = rdf.AsNumpy(['track_pt'])

expected_sum = np.array([np.sum(pt) for pt in raw_all['track_pt']])
expected_mean = np.array([np.mean(pt) if len(pt) > 0 else 0 for pt in raw_all['track_pt']])
expected_max = np.array([np.max(pt) if len(pt) > 0 else 0 for pt in raw_all['track_pt']])
expected_min = np.array([np.min(pt) if len(pt) > 0 else 0 for pt in raw_all['track_pt']])

assert np.allclose(df['sum_pt'], expected_sum), "Sum mismatch"
assert np.allclose(df['mean_pt'], expected_mean), "Mean mismatch"
assert np.allclose(df['max_pt'], expected_max), "Max mismatch"
assert np.allclose(df['min_pt'], expected_min), "Min mismatch"

print("✓ 1D reductions: All validations passed")
df.head()

## Section 3: 1D Element Access

### Description
Test single element access from RVec.

### What we're testing
- First element: `track_pt[0]`
- Last element: `track_pt[-1]`
- Second element: `track_pt[1]`

In [ ]:
dsl3 = DSLCompiler(schema)

dsl3.define("first", "track_pt[0]")
dsl3.define("last", "track_pt[-1]")
dsl3.define("second", "track_pt[1]")

df3 = dsl3.to_pandas(rdf, ['first', 'last', 'second'])

# Validate
for i in range(min(10, len(raw_all['track_pt']))):
    pt = np.array(raw_all['track_pt'][i])  # Convert RVec to numpy
    if len(pt) > 0:
        assert np.isclose(df3['first'].iloc[i], pt[0]), f"Event {i}: first mismatch"
        assert np.isclose(df3['last'].iloc[i], pt[-1]), f"Event {i}: last mismatch"
    if len(pt) > 1:
        assert np.isclose(df3['second'].iloc[i], pt[1]), f"Event {i}: second mismatch"

print("✓ 1D element access: All validations passed")

## Section 4: 1D Transformations

### Description
Test element-wise operations on RVec.

### What we're testing
- Scalar multiplication: `track_pt / 1000` (GeV to TeV)
- Functions: `sqrt(track_pt)`
- Combined: `(track_pt - Mean(track_pt)) / sqrt(Sum((track_pt - Mean(track_pt))**2))`

In [ ]:
dsl4 = DSLCompiler(schema)

dsl4.define("pt_tev", "track_pt / 1000")
dsl4.define("sqrt_pt", "sqrt(track_pt)")
dsl4.define("pt_squared", "track_pt ** 2")

applied4 = dsl4.apply(rdf)
result4 = applied4.Range(5).AsNumpy(['track_pt', 'pt_tev', 'sqrt_pt', 'pt_squared'])

# Validate
for i in range(5):
    pt = np.array(result4['track_pt'][i])  # Convert RVec to numpy
    
    assert np.allclose(np.array(result4['pt_tev'][i]), pt / 1000), f"Event {i}: pt_tev mismatch"
    assert np.allclose(np.array(result4['sqrt_pt'][i]), np.sqrt(pt)), f"Event {i}: sqrt_pt mismatch"
    assert np.allclose(np.array(result4['pt_squared'][i]), pt ** 2), f"Event {i}: pt_squared mismatch"

print("✓ 1D transformations: All validations passed")

## CPU Benchmarking

Compare DSL vs direct computation for 1D operations.

In [ ]:
def benchmark(name, func, n_runs=3):
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        result = func()
        times.append(time.perf_counter() - start)
    print(f"{name}: {np.mean(times):.4f}s ± {np.std(times):.4f}s")
    return result, times

# DSL Sum reduction
dsl_bench = DSLCompiler(schema)
dsl_bench.define("sum_pt", "Sum(track_pt)")
_, times_dsl = benchmark("DSL Sum(track_pt)", 
                         lambda: dsl_bench.to_pandas(rdf, ['sum_pt']))

# Python loop equivalent
def python_sum():
    data = rdf.AsNumpy(['track_pt'])
    return np.array([np.sum(pt) for pt in data['track_pt']])

_, times_python = benchmark("Python loop", python_sum)

print(f"\nDSL speedup: {np.mean(times_python)/np.mean(times_dsl):.2f}x")

## Summary

In [ ]:
print("=" * 50)
print("08b_debug_1d.ipynb - L2 1D Array Operations")
print("=" * 50)
print("\n✓ Section 1: 1D slicing - PASSED")
print("✓ Section 2: 1D reductions - PASSED")
print("✓ Section 3: 1D element access - PASSED")
print("✓ Section 4: 1D transformations - PASSED")
print("\n" + "=" * 50)
print("ALL TESTS PASSED")
print("=" * 50)